# Trotterized Quantum Simulation of a Periodic Heisenberg Chain

This notebook follows the 2026 Quantum Korea Hackathon problem set on Trotterized time evolution.

We study the periodic anisotropic Heisenberg chain with a transverse X field,

$$
H = \sum_{j=0}^{N-1}\left(J_x X_jX_{j+1}+J_yY_jY_{j+1}+J_zZ_jZ_{j+1}\right)
+ h_x\sum_{j=0}^{N-1}X_j,
$$

where $j+1$ is understood modulo $N$.

Default parameters:

$$
N=6,\quad J_x=1.0,\quad J_y=0.7,\quad J_z=1.2,\quad h_x=0.4.
$$

The initial state is

$$
|\psi_0\rangle = |010101\rangle,
$$

where the bit string is written in **qubit-0-first** convention: the first character is qubit 0.

The time grid is

$$
t = 0,0.05,0.10,\ldots,15.0.
$$

## Setup and Imports

In [ ]:
# Uncomment if needed.
# %pip install -r ./requirements.txt

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.synthesis import ProductFormula, LieTrotter, SuzukiTrotter

# Import anything you need. We recommend using only libraries listed in requirements.txt.

In [ ]:
N = 6
Jx = 1.0
Jy = 0.7
Jz = 1.2
hx = 0.4

BITSTRING_NEEL = "101010"
DT_GRID = 0.05
T_MAX = 15.0
TIMES = np.arange(0.0, T_MAX + 0.5 * DT_GRID, DT_GRID)

BASIS_GATES = ["cx", "u3", "u1"]
K_VALUES = [1, 2, 3]

print(f"Number of time points: {len(TIMES)}")
print(f"Initial state, qubit-0-last: |{BITSTRING_NEEL}>")

## Problem 1(a): construct the Hamiltonian

Build the `SparsePauliOp` Hamiltonian, report the total number of Pauli terms, and identify terms introduced by the periodic boundary condition.

**Hint:** Refer to the following figure to optimize the circuit depth and expected Trotter error. Explain which specific term ordering is expected to give lower circuit depth and Trotter error, and why.
![ham_ordering.png](ham_ordering.png)

In [ ]:
def build_heisenberg_hamiltonian(
        num_qubits:int=N, Jx:float=Jx, Jy:float=Jy, Jz:float=Jz, hx:float=hx
    )->SparsePauliOp:
    """Build the periodic Heisenberg-chain Hamiltonian as a SparsePauliOp.

    TODO:
    1. Add XX, YY, ZZ terms for each edge (j, (j+1) mod N).
    2. Mark the periodic-boundary terms, where (j+1) mod N == 0.
    3. Add transverse X-field terms.
    4. Return both the SparsePauliOp and a readable term table.
    """

    return hamiltonian

In [ ]:
H_spo = build_heisenberg_hamiltonian()

# TODO: report the total number of terms.
# TODO: identify the periodic-boundary terms.

## Exact reference simulation

Since $N=6$, exact classical simulation is cheap. Use it as the reference for state and observable errors.

In [ ]:
psi0 = Statevector.from_label(BITSTRING_NEEL)
H_dense = H_spo.to_matrix()

# TODO: diagonalize H_dense.
# TODO: precompute the expansion coefficients of psi0 in the eigenbasis.

def exact_state(time: float) -> Statevector:
    """Return exp(-i H t)|psi0> from exact diagonalization."""
    # TODO: Implementa and use this function for the exact simulation
    raise NotImplementedError


def observable_matrix(pauli_label: str) -> SparsePauliOp:
    return SparsePauliOp.from_list([(pauli_label, 1.0)]).to_matrix()


Z1_label = "".join(["Z" if i in [1] else "I" for i in range(N)])
Z0Z1_label = "".join(["Z" if i in [0, 1] else "I" for i in range(N)])
Z1 = observable_matrix(Z1_label)
Z0Z1 = observable_matrix(Z0Z1_label)

# TODO: compute exact_states, exact_z1, exact_z0z1 on TIMES.

## Problem 1(b-c): product-formula circuits and resource counts

Construct circuits for each target time using `PauliEvolutionGate` with `LieTrotter(reps=k)`, `SuzukiTrotter(order=2, reps=k)`, and `SuzukiTrotter(order=4, reps=k)` for $k=1,2,3$. Then transpile into `{cx, u3, u1}` and report resource counts.

In [ ]:
def synthesis_settings(k_values=K_VALUES) -> Dict[str, ProductFormula]:
    settings = {}
    for k in k_values:
        settings[f"LT_k{k}"] = LieTrotter(reps=k)
        settings[f"ST2_k{k}"] = SuzukiTrotter(order=2, reps=k)
        settings[f"ST4_k{k}"] = SuzukiTrotter(order=4, reps=k)
    return settings


SETTINGS = synthesis_settings()


def evolution_circuit(time:float, synthesis:ProductFormula, include_stateprep=False) -> QuantumCircuit:
    """Build a circuit containing a synthesized approximation to exp(-iHt)."""
    qc = QuantumCircuit(N)
    if include_stateprep:
        # TODO: prepare the state defined by BITSTRING_NEEL.
    # TODO: append pauli evolution gates
    return qc


def transpile_to_prompt_basis(qc:QuantumCircuit, optimization_level:int=0) -> QuantumCircuit:
    # TODO: transpile into BASIS_GATES = ["cx", "u3", "u1"].
    return transpiled_qc


def product_formula_state(time:float, synthesis:ProductFormula) -> Statevector:
    # TODO: build state-prep + time-evolution circuit, and return Statevector.from_instruction(...).data.
    return trotter_state

In [ ]:
def resource_counts(qc:QuantumCircuit) -> Dict[str, int]:
    circuit_summary = {
        "gate count": None
        "two-qubit gates": None,
        "one-qubit gates": None,
        "depth": None
    }
    # TODO: return depth, total gate count, number of cx gates, and number of one-qubit gates.
    return ops


t_resource = 0.5
rows = []
for name, synth in SETTINGS.items():
    # TODO: build evolution-only circuit, transpile it, and append resource counts to rows.

resource_table = pd.DataFrame(rows)
display(resource_table)

## Problem 1(d-f): state and local-observable errors

Compare the product-formula state to the exact state using

$$\epsilon_{state}(t)=1-|\langle\psi_{exact}(t)|\psi_{PF}(t)
\rangle|^2.$$ 

For local observables $Z_1$ and $Z_0Z_1$, compute

$$\epsilon_O(t)=|\langle O
\rangle_{exact}(t)-\langle O
\rangle_{PF}(t)|.$$

In [ ]:
def state_infidelity(exact:Statevector, approx:Statevector) -> float:
    # TODO: return 1 - |<exact|approx>|^2.
    return infidelity


error_rows = []
for setting_name, synth in SETTINGS.items():
    for time, st_exact, z1_exact, z0z1_exact in zip(TIMES, exact_states, exact_z1, exact_z0z1):
        # TODO:
        # 1. compute the product-formula state,
        # 2. compute state infidelity,
        # 3. compute Z1 and Z0Z1 local-observable errors,
        # 4. append results to error_rows.

error_table = pd.DataFrame(error_rows)
display(error_table.head(n=5))

In [ ]:
# TODO: plot time versus state_infidelity, Z1_error, and Z0Z1_error.

## Problem 1(g): Run on an IBM backend

For this problem, set $N=12$, $J_x=J_y=1.0, J_z=1.2,$ and $h_x=0$. Use the initial state as 
$$
|\psi\rangle = |1000 0000 0000\rangle.
$$

Choose the simulation time as $t=1.0$ and estimate $\langle Z_0\rangle$ on an available IBM Heron backend.
Use Qiskit Runtime error mitigation by setting `resilience_level=2`, and compare the hardware results with the ideal exact values.

**Hint 1**: The interaction graph of a periodic (N=12) Heisenberg model can be embedded in the connectivity of IBM Heron processors. Consider placing the circuit on an appropriate cycle in the QPU coupling map to avoid adding `SWAP` gates or routing in the transpilation phase.

**Hint 2**: When $J_x=J_y$ and $h_x=0$, the Hamiltonian preserves the number of $1$s in the computational basis. In other words,
$$
[H, N] = 0,\quad \text{for }N=\sum_i \frac{1-Z_i}{2}.
$$
Because the initial state contains a single (1), the ideal time evolution satisfies
$$
\langle\psi(t)|N|\psi(t)\rangle = 1 \quad\text{and}\quad |\psi(t)\rangle \in \mathrm{span}\{|1000 0000 0000\rangle, |0100 0000 0000\rangle, \cdots |0000 0000 0001\rangle\}.
$$
Therefore, the dynamics are restricted to an effective Hilbert space of dimension $12$, which is much smaller than the full Hilbert-space dimension $2^{12}$.

This symmetry can be used to calculate the ideal values efficiently on a classical computer.
The `single_one_evolution_meas_Z0()` utility implements this calculation. Use its output as a baseline for comparison.

Note that a Néel state still has a much larger effective subspace, even after restricting the dynamics using this symmetry.

**Hint 3**: Before running the circuit, confirm that the transpiled circuit to have
$$
\text{depth} < 100 \quad \text{two qubit gate count} < 400\quad \text{two qubit depth} <30.
$$


In [ ]:
# TODO: Choose a time point from TIMES.
# TODO: Select one product-formula setting, for example SuzukiTrotter(order=2, reps=1 or 2).
# TODO: Build the corresponding Trotterized circuits.
# TODO: Select an available IBM backend.
# TODO: Transpile the circuits for the backend.
# TODO: Apply the final transpiler layout to the observables.
# TODO: Use EstimatorV2 with resilience_level=2 to estimate <Z1> and <Z0Z1>.
# TODO: Compare the backend estimates with the ideal exact values.

****Challenge****: Maintaining the settings above, increase $N=1+8r$ and set the initial state as 

$$
|\psi\rangle = |1\otimes 0^{\otimes N-1}\rangle,
$$

where the integer $r$ is the number of heavy-hex lattice of Heron processors ($r=1$ corresponds to the problem above).

Estimate $\langle Z_0\rangle$ for a single time step that you choose on an available IBM backend.

Solve the next the problems and come back to scale up the $r$.

In [ ]:
# TODO: Your code here

## Problem 1(h): discussion prompt

Use your resource table and error plots to answer which formula is most accurate at the same repetition number, which formula is best at comparable circuit depth, and what trade-off is observed between depth and approximation error.

## Problem 2(a): exact autocorrelation

Return back to the small size problem and settings in the heading.

The autocorrelation signal is

$$A(t)=\langle\psi_0|e^{-iHt}|\psi_0\rangle.$$

In [ ]:
# TODO: compute A_exact(t) = <psi0|exp(-iHt)|psi0> and plot its real part, imaginary part, and absolute value.

## Problem 2(b-c): Trotterized autocorrelation

Construct a Hadamard-test circuit to compute

$$A_{PF}(t)=\langle\psi_0|\psi_{PF}(t)
\rangle.$$

In [ ]:
# TODO: choose at least one product-formula setting from Problem 1.
# Compute A_PF(t) by a Hadamard-test circuit with local simulator.
# Plot |A_exact(t) - A_PF(t)|.

## Problem 2(d-e): Fourier spectrum

Apply a DFT to $A(t)$ and identify dominant peaks. Use the eigenbasis expression of $A(t)$ to explain why peak locations correspond to eigenenergies with nonzero overlap with the initial state.

In [ ]:
# TODO: apply a discrete Fourier transform to A_exact(t).
# Suggested convention:
#     spectrum = fftshift(fft(A_exact * window))
#     freq = fftshift(fftfreq(len(A_exact), d=DT_GRID))
#     energy_axis = -2*pi*freq
# because A(t) = sum_n |<E_n|psi0>|^2 exp(-i E_n t).
# Plot the spectrum and identify dominant peaks.

## Bonus Problem 3 and optional Rustiq-based synthesis

The Rustiq comparison depends on the Qiskit version and installed optional packages in your environment. If Rustiq synthesis is available, use the same representative circuit and compare the resulting depth and two-qubit gate count against the standard transpilation table above. If it is not available, state explicitly that the environment did not provide Rustiq support.

In [ ]:
# Bonus TODO:
# 1. Pick one representative circuit, e.g. SuzukiTrotter(order=2, reps=3), t=0.5.
# 2. Transpile it for a fake or real backend.
# 3. Compare optimization levels 0, 1, 2, 3.
# 4. Report depth, total gate count, and two-qubit gate count.